<a href="https://colab.research.google.com/github/Pensive1881/DSR44_2025/blob/main/clip_plus_detection_and_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%%capture
!uv pip install fiftyone==1.9.0

## Obtain the CLIP model

In [5]:
import fiftyone as fo
import fiftyone.zoo as foz

clip_model = foz.load_zoo_model("clip-vit-base32-torch")

In [6]:
from google.colab import drive
drive.mount('/gdrive')
%cd /gdrive

Mounted at /gdrive
/gdrive


## Create the FiftyOne dataset with our custom artwork

In [14]:
from pathlib import Path
import os
artist_name = 'mark ryden'
path = path = Path(f'/gdrive/MyDrive/art_recommendation/{artist_name}')

In [15]:
# here 'paintings' and 'paintings_embeddings.pickle' should both appear
os.listdir(path)

['paintings', 'mark ryden illustration']

In [19]:
len(os.listdir(path / 'paintings'))

178

In [20]:
import fiftyone as fo

dataset_name = dataset_name = f"{artist_name}_paintings"

# delete the dataset in case it exists already on the Colab instance
# (due to multiple evaluations of the code cell)
if fo.dataset_exists(dataset_name):
  fo.delete_dataset(dataset_name)

In [21]:
# this creates an empty dataset
dataset = fo.Dataset(dataset_name)
dataset

Name:        mark ryden_paintings
Media type:  None
Num samples: 0
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.Metadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField

In [23]:
images_dir = path / 'paintings'

In [24]:
# We use the location of the images_dir to add samples to the dataset
dataset.add_dir(images_dir, dataset_type=fo.types.ImageDirectory)

 100% |█████████████████| 178/178 [65.3ms elapsed, 0s remaining, 2.7K samples/s]   


INFO:eta.core.utils: 100% |█████████████████| 178/178 [65.3ms elapsed, 0s remaining, 2.7K samples/s]   


['69037b118f8f808eaf49ca00',
 '69037b118f8f808eaf49ca01',
 '69037b118f8f808eaf49ca02',
 '69037b118f8f808eaf49ca03',
 '69037b118f8f808eaf49ca04',
 '69037b118f8f808eaf49ca05',
 '69037b118f8f808eaf49ca06',
 '69037b118f8f808eaf49ca07',
 '69037b118f8f808eaf49ca08',
 '69037b118f8f808eaf49ca09',
 '69037b118f8f808eaf49ca0a',
 '69037b118f8f808eaf49ca0b',
 '69037b118f8f808eaf49ca0c',
 '69037b118f8f808eaf49ca0d',
 '69037b118f8f808eaf49ca0e',
 '69037b118f8f808eaf49ca0f',
 '69037b118f8f808eaf49ca10',
 '69037b118f8f808eaf49ca11',
 '69037b118f8f808eaf49ca12',
 '69037b118f8f808eaf49ca13',
 '69037b118f8f808eaf49ca14',
 '69037b118f8f808eaf49ca15',
 '69037b118f8f808eaf49ca16',
 '69037b118f8f808eaf49ca17',
 '69037b118f8f808eaf49ca18',
 '69037b118f8f808eaf49ca19',
 '69037b118f8f808eaf49ca1a',
 '69037b118f8f808eaf49ca1b',
 '69037b118f8f808eaf49ca1c',
 '69037b118f8f808eaf49ca1d',
 '69037b118f8f808eaf49ca1e',
 '69037b118f8f808eaf49ca1f',
 '69037b118f8f808eaf49ca20',
 '69037b118f8f808eaf49ca21',
 '69037b118f8f

In [25]:
# add metadata on file size, image format, and image dimensions
dataset.compute_metadata()

Computing metadata...


INFO:fiftyone.core.metadata:Computing metadata...


 100% |█████████████████| 178/178 [2.9s elapsed, 0s remaining, 201.8 samples/s]     


INFO:eta.core.utils: 100% |█████████████████| 178/178 [2.9s elapsed, 0s remaining, 201.8 samples/s]     


## Compute CLIP embeddings

In [34]:
import fiftyone.brain as fob

image_index = fob.compute_similarity(
    dataset,
    model="clip-vit-base32-torch",
    brain_key="clip_img_sim",
    progress_bar=True,
    embeddings='clip_embeddings'
)

In [35]:
info = dataset.get_brain_info("clip_img_similarity")
print(info.config.supports_prompts)  # Should print True

None


In [36]:
session = fo.launch_app(dataset, auto=False)
print(session.url)

Session launched. Run `session.show()` to open the App in a cell output.


INFO:fiftyone.core.session.session:Session launched. Run `session.show()` to open the App in a cell output.


https://5151-m-s-3p8qzmxzex6m0-c.us-east1-0.prod.colab.dev?polling=true


## Open vocabulary detection and segmentation with YOLOE

In [38]:
%%capture
!uv pip install ultralytics

In [39]:
from ultralytics import YOLOE
import os
import requests
from tqdm.auto import tqdm

model_path = "yoloe-11s-seg.pt"
if not os.path.exists(model_path):
    url = "https://github.com/ultralytics/assets/releases/download/v8.3.0/yoloe-11s-seg.pt"
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    block_size = 1024 # 1 KB
    tqdm_bar = tqdm(total=total_size, unit='iB', unit_scale=True)
    with open(model_path, 'wb') as f:
        for data in response.iter_content(block_size):
            tqdm_bar.update(len(data))
            f.write(data)
    tqdm_bar.close()

model = YOLOE(model_path)

names = ["a boat full of people", "a boat", 'people', 'fish', 'octopus']
model.set_classes(names, model.get_text_pe(names))

dataset.apply_model(model, label_field="yoloe_segmentation")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


  0%|          | 0.00/27.8M [00:00<?, ?iB/s]

OSError: [Errno 95] Operation not supported: 'yoloe-11s-seg.pt'

In [41]:
from ultralytics import YOLOE
import os

model_path = "yoloe-11s-seg.pt"

model = YOLOE(model_path)

names = ["a boat full of people", "a boat", 'people', 'fish', 'octopus']
model.set_classes(names, model.get_text_pe(names))

dataset.apply_model(model, label_field="yoloe_segmentation")

WARNING ⚠️ Download failure, retrying 1/3 https://github.com/ultralytics/assets/releases/download/v8.3.0/yoloe-11s-seg.pt... [Errno 95] Operation not supported: 'yoloe-11s-seg.pt'
WARNING ⚠️ Download failure, retrying 2/3 https://github.com/ultralytics/assets/releases/download/v8.3.0/yoloe-11s-seg.pt... Curl return value 23
WARNING ⚠️ Download failure, retrying 3/3 https://github.com/ultralytics/assets/releases/download/v8.3.0/yoloe-11s-seg.pt... Curl return value 23


ConnectionError: ❌  Download failure for https://github.com/ultralytics/assets/releases/download/v8.3.0/yoloe-11s-seg.pt. Retry limit reached. Curl return value 23

In [42]:
# Commented out IPython magic to ensure Python compatibility.
from google.colab import drive
drive.mount('/gdrive')

from pathlib import Path
from ultralytics import YOLOE
import shutil

# Define paths - note we're using /content (local) not /gdrive
local_model_path = Path("/content/yoloe-11s-seg.pt")
gdrive_model_path = Path('/gdrive/MyDrive/YOLOE/yoloe-11s-seg.pt')

# Create the Drive directory if it doesn't exist
gdrive_model_path.parent.mkdir(parents=True, exist_ok=True)

# Check if model exists in Drive, if so copy to local
if gdrive_model_path.exists():
    print("Copying model from Google Drive to local storage...")
    shutil.copy(gdrive_model_path, local_model_path)
    model = YOLOE(str(local_model_path))
else:
    # Download to local storage (this will work)
    print("Downloading model to local storage...")
    model = YOLOE("yoloe-11s-seg.pt")  # Downloads to local

    # Copy to Drive for persistence
    print("Saving model to Google Drive for future use...")
    shutil.copy(local_model_path, gdrive_model_path)

# Continue with your code
names = ["a boat full of people", "a boat", 'people', 'fish', 'octopus']
model.set_classes(names, model.get_text_pe(names))

dataset.apply_model(model, label_field="yoloe_segmentation")

Drive already mounted at /gdrive; to attempt to forcibly remount, call drive.mount("/gdrive", force_remount=True).
WARNING ⚠️ Download failure, retrying 1/3 https://github.com/ultralytics/assets/releases/download/v8.3.0/yoloe-11s-seg.pt... [Errno 95] Operation not supported: 'yoloe-11s-seg.pt'
WARNING ⚠️ Download failure, retrying 2/3 https://github.com/ultralytics/assets/releases/download/v8.3.0/yoloe-11s-seg.pt... Curl return value 23
WARNING ⚠️ Download failure, retrying 3/3 https://github.com/ultralytics/assets/releases/download/v8.3.0/yoloe-11s-seg.pt... Curl return value 23


ConnectionError: ❌  Download failure for https://github.com/ultralytics/assets/releases/download/v8.3.0/yoloe-11s-seg.pt. Retry limit reached. Curl return value 23